# Beadform expanded free Qwen run (v2)

1. Choose **Runtime → Change runtime type → T4 GPU** using the free tier.
2. Run **setup** and connect the same Google Drive if requested.
3. Run **training**. Leave the notebook open while it runs.

This run uses **320 synthetic training requests**, the original **48 development requests**, and a frozen **80-request synthetic challenge**. It selects one of 32 existing draft subjects or `unknown`; fish stays excluded. It does not generate new bead geometry.

The challenge is not used for training or choosing epochs. Compare the base model, the preserved v1 adapter, and the new v2 adapter on the same requests. These authored examples are not an independent user benchmark or proof of beadability.

Saves separately to `MyDrive/Beadform-free-qwen/runs/catalog-selector-v2`; v1 is preserved. Same frozen Qwen model revision, fixed two epochs, language-attention QLoRA only, no automatic deployment. No images, physical-success labels, paid services or quota workarounds. Free GPU availability varies; stop if no GPU is available.


In [ ]:
import sys, subprocess, json, hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Free GPU unavailable. Stop here and retry later; no paid upgrade.")
from google.colab import drive
drive.mount('/content/drive')
run = Path('/content/drive/MyDrive/Beadform-free-qwen/runs/catalog-selector-v2')
previous = run.parent/'catalog-selector-v1'
if not (previous/'comparison.json').exists():
    raise RuntimeError("The completed v1 run must remain in Drive for the comparison.")
run.mkdir(parents=True, exist_ok=True)
repo = Path('/content/beadform-catalog-selector-v2')
if not repo.exists():
    subprocess.run(['git','clone','--branch','runpod-training-guide','https://github.com/c-tustin/beadform.git',str(repo)],check=True)
if subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=repo,text=True).strip():
    raise RuntimeError("Source has edits; preserve them before continuing.")
revision_file=run/'repository-revision.txt'
if revision_file.exists():
    revision=revision_file.read_text().strip()
else:
    revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
    revision_file.write_text(revision+'\n')
subprocess.run(['git','fetch','origin',revision],cwd=repo,check=True)
subprocess.run(['git','checkout','--detach',revision],cwd=repo,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'training/requirements-catalog-v2.txt')],check=True)
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
freeze_file=run/'pip-freeze.txt'
if freeze_file.exists() and freeze_file.read_text()!=freeze:
    raise RuntimeError("Environment changed. Preserve this run; inspect before resuming.")
freeze_file.write_text(freeze)
print('Repository',revision,'GPU',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,str(repo/'training/train_catalog_selector_v2.py'),'--output',str(run),'--previous-run',str(previous),'--check'],cwd=repo,check=True)


In [ ]:
command=[sys.executable,'-u',str(repo/'training/train_catalog_selector_v2.py'),'--output',str(run),'--previous-run',str(previous),'--epochs','2','--resume']
if not (run/'comparison.json').exists():
    with (run/'run.log').open('a') as log:
        process=subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        code=process.wait()
        if code:
            raise RuntimeError(f'Training stopped with code {code}; inspect the saved log.')
print((run/'comparison.json').read_text())
print('Saved:',run)
print('Synthetic catalog selection only. No automatic deployment or physical validation.')
